In [13]:
"""
ContextBuilder 与 Agent 集成示例

展示如何将 ContextBuilder 集成到 Agent 中，实现：
1. 上下文感知的 Agent
2. 自动构建优化的上下文
3. 记忆管理与上下文构建的协同
"""
from dotenv import load_dotenv
env_path = "/t9k/mnt/lxq/hello_agents/.env"
_ = load_dotenv(dotenv_path=env_path)
from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry
from hello_agents.context import ContextBuilder, ContextConfig
from hello_agents.tools import MemoryTool, RAGTool
from hello_agents.core.message import Message
from datetime import datetime

In [14]:
class ContextAwareAgent(SimpleAgent):
    """具有上下文感知能力的 Agent"""

    def __init__(self, name: str, llm: HelloAgentsLLM, **kwargs):
        super().__init__(name=name, llm=llm, **kwargs)

        
        #（Optional）
        # self.memory_tool = MemoryTool(user_id=kwargs.get("user_id", "default")) 
        # self.rag_tool = RAGTool(knowledge_base_path=kwargs.get("knowledge_base_path", "./kb"))

        # 初始化上下文构建器
        self.context_builder = ContextBuilder(
            # memory_tool=self.memory_tool,
            # rag_tool=self.rag_tool,
            config=ContextConfig(max_tokens=4000)
        )

        self.conversation_history = []

    def run(self, user_input: str) -> str:
        """运行 Agent,自动构建优化的上下文"""

        # 1. 使用 ContextBuilder 构建优化的上下文
        optimized_context = self.context_builder.build(
            user_query=user_input,
            conversation_history=self.conversation_history,
            system_instructions=self.system_prompt
        )

        # 2. 使用优化后的上下文调用 LLM
        messages = [
            {"role": "system", "content": optimized_context},
            {"role": "user", "content": user_input}
        ]
        response = self.llm.invoke(messages)

        # 3. 更新对话历史
        self.conversation_history.append(
            Message(content=user_input, role="user", timestamp=datetime.now())
        )
        self.conversation_history.append(
            Message(content=response, role="assistant", timestamp=datetime.now())
        )

        # 4. 将重要交互记录到记忆系统
        # self.memory_tool.run({
        #     "action": "add",
        #     "content": f"Q: {user_input}\nA: {response[:200]}...",  # 摘要
        #     "memory_type": "episodic",
        #     "importance": 0.6
        # })

        return response

In [16]:
def main():
    print("=" * 80)
    print("ContextBuilder 与 Agent 集成示例")
    print("=" * 80 + "\n")

    # 配置 LLM
    from hello_agents.core.llm import HelloAgentsLLM
    llm = HelloAgentsLLM()

    # 使用示例
    agent = ContextAwareAgent(
        name="数据分析顾问",
        llm=llm,
        system_prompt="你是一位资深的Python数据工程顾问。"
    )

    # 进行对话
    response = agent.run("如何优化Pandas的内存占用?")
    print(f"助手回答:\n{response}\n")

    # 继续对话
    response = agent.run("能给出具体的代码示例吗?")
    print(f"助手回答:\n{response}\n")

    print("=" * 80)

In [17]:
if __name__ == "__main__":
    main()

ContextBuilder 与 Agent 集成示例



INFO:openai._base_client:Retrying request to /chat/completions in 0.383166 seconds


助手回答:
1. 结论
优化Pandas的内存占用可以通过减少数据类型大小、使用类别数据类型、删除不必要的列或行以及利用更高效的数据结构来实现。这些方法有助于显著降低程序运行时所消耗的内存，提高处理大数据集的能力。

2. 依据
- 减少数据类型大小：对于整数和浮点数列，如果它们的实际值范围远小于默认类型的范围（例如int64, float64），可以将它们转换为更小的数据类型（如int8, int16, float32）。这一步骤可以直接减少每个元素所需的存储空间。
- 使用类别数据类型：当一列包含有限数量的不同值时，可以将其转换为categorical类型，这样不仅减少了内存使用量，还可能加快某些操作的速度。
- 删除不必要的列或行：去除那些不参与计算或者分析的冗余信息，直接减少了DataFrame的大小。
- 利用更高效的数据结构：考虑使用其他专门针对特定场景优化过的库，比如Dask、Vaex等，它们能够更好地处理大规模数据集，并且往往具有更好的性能表现。
  
参考文献：
- [Pandas官方文档](https://pandas.pydata.org/pandas-docs/stable/user_guide/index.html#user-guide)
- [Efficient Data Structures in Python](https://realpython.com/efficient-data-structures-python/)

3. 风险与假设
- 在转换数据类型时需要注意不要丢失重要信息，特别是从高精度转到低精度的过程中可能会导致数值截断。
- 将字符串转换为category类型虽然能节省内存，但如果之后需要对这些值进行修改，则会比较麻烦，因为category是不可变的。
- 使用第三方库如Dask、Vaex等可能引入额外的学习成本和技术债务。

4. 下一步行动建议
- 对当前项目中的所有DataFrame进行全面审查，识别出可以应用上述优化策略的地方。
- 实施更改前先备份原始数据，确保任何意外情况发生时都能恢复。
- 考虑编写自动化脚本来定期检查并调整DataFrame以维持最佳性能状态。
- 如果遇到特别大的数据集难以处理的情况，评估是否有必要转向分布式计算框架或其他更适合的解决方案。

助手回答:
您的问题比较宽泛，没有具